# Bluebikes Monthly Ridership, Year over Year

Pulls monthly trip-history data from the [Bluebikes/Hubway System Data bucket](https://s3.amazonaws.com/hubway-data/index.html) for a specified `YYYY-MM` to `YYYY-MM` range, plus the same months one year prior, and overlays both years' monthly ridership as lines aligned by calendar month.

Trip counts are taken as a row count per month's CSV, which is schema-agnostic across the older Hubway column layout and the current one, this only needs ridership volume, not any per-trip fields.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from matplotlib.ticker import FuncFormatter

from shared.plots.style import apply_theme, add_source_footnote

apply_theme()

## Params

In [ ]:
MONTH_START = "2025-04"  # YYYY-MM, inclusive
MONTH_END = "2026-08"  # YYYY-MM, inclusive

## Load or fetch monthly trip counts

Fetches the requested range plus the same months one year earlier, so the two years can be overlaid by calendar month. Each month is cached independently at `data/raw/bluebikes_tripdata/{yyyymm}/data.csv`, so re-running the notebook (or widening the range) only fetches the months not already on disk.

In [ ]:
BASE_URL = "https://s3.amazonaws.com/hubway-data"


def fetch_monthly_counts(month_start, month_end):
    months = pd.period_range(month_start, month_end, freq="M")

    records = []
    for period in months:
        yyyymm = period.strftime("%Y%m")
        raw_path = Path(f"data/raw/bluebikes_tripdata/{yyyymm}/data.csv")

        if not raw_path.exists():
            # Almost every month is "{yyyymm}-bluebikes-tripdata.zip"; a rare
            # month (e.g. 202511) ships as "{yyyymm}-bluebikes-tripdata.csv.zip" instead.
            for suffix in ("-bluebikes-tripdata.zip", "-bluebikes-tripdata.csv.zip"):
                url = f"{BASE_URL}/{yyyymm}{suffix}"
                response = requests.get(url, timeout=60)
                if response.ok:
                    break
            response.raise_for_status()

            with ZipFile(BytesIO(response.content)) as zf:
                csv_name = next(
                    name
                    for name in zf.namelist()
                    if name.endswith(".csv") and "__MACOSX" not in name
                )
                raw_path.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(csv_name) as src, open(raw_path, "wb") as dst:
                    dst.write(src.read())

            print(f"Fetched and cached: {raw_path}")
        else:
            print(f"Loaded cached pull: {raw_path}")

        trip_count = sum(1 for _ in open(raw_path)) - 1  # rows minus header
        records.append({"month": period.to_timestamp(), "trip_count": trip_count})

    return pd.DataFrame(records)


prior_month_start = (pd.Period(MONTH_START, freq="M") - 12).strftime("%Y-%m")
prior_month_end = (pd.Period(MONTH_END, freq="M") - 12).strftime("%Y-%m")

current_year = fetch_monthly_counts(MONTH_START, MONTH_END)
prior_year = fetch_monthly_counts(prior_month_start, prior_month_end)

# Align each prior-year month under the current-year month it precedes by
# exactly 12 months, so the two lines share an x-axis position per calendar month.
prior_year_aligned = prior_year.rename(columns={"trip_count": "prior_trip_count"})
prior_year_aligned["month"] = prior_year_aligned["month"] + pd.DateOffset(years=1)

monthly = current_year.merge(prior_year_aligned[["month", "prior_trip_count"]], on="month", how="left")
monthly.head()

## Chart: monthly ridership, year over year

In [ ]:
range_label = f"{MONTH_START}_{MONTH_END}"
start_label = pd.Period(MONTH_START, freq="M").strftime("%b %Y")
end_label = pd.Period(MONTH_END, freq="M").strftime("%b %Y")

current_label = f"{start_label} to {end_label}"
prior_start_label = pd.Period(prior_month_start, freq="M").strftime("%b %Y")
prior_end_label = pd.Period(prior_month_end, freq="M").strftime("%b %Y")
prior_label = f"{prior_start_label} to {prior_end_label}"

plot_data = monthly.melt(
    id_vars="month",
    value_vars=["trip_count", "prior_trip_count"],
    var_name="series",
    value_name="trips",
)
plot_data["series"] = plot_data["series"].map(
    {"trip_count": current_label, "prior_trip_count": prior_label}
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(
    data=plot_data,
    x="month",
    y="trips",
    hue="series",
    hue_order=[prior_label, current_label],
    marker="o",
    ax=ax,
)

ax.set_title(f"Bluebikes Monthly Ridership, {current_label} vs prior year")
ax.set_xlabel("")
ax.set_ylabel("Trips")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(title="")
fig.autofmt_xdate()

add_source_footnote(fig, "Bluebikes / Hubway System Data (s3.amazonaws.com/hubway-data)")

output_dir = Path(f"outputs/{range_label}/charts")
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / "monthly_ridership_yoy.png", bbox_inches="tight")
plt.show()